# One-run class-balanced sampling experiment
Attach BTP-code-sampling.zip, quarter_lr_to_epoch100_20260918_094320.zip and prepared public252 data. Extraction is automatic. This trains ONE new arm for epochs101-110, reusing the verified original control records bundled with the code. Allow approximately1hour plus setup/archive time. Keep the original checkpoints and download the complete new output ZIP.

In [ ]:
from pathlib import Path
import shutil, zipfile, subprocess, sys
INPUTS = Path('/kaggle/input')
CODE = Path('/kaggle/working/BTP-sampling')
def safe_extract(z, dest):
    dest = dest.resolve()
    for member in z.infolist():
        if not (dest / member.filename).resolve().is_relative_to(dest):
            raise ValueError('Unsafe ZIP member')
    z.extractall(dest)
# Identify this evaluator by its actual feature, not a shared filename.
import tempfile
marker = 'Single balanced-sampling arm against the archived epoch101-110 control'
def is_audit_source(path):
    return path.is_file() and marker in path.read_text()
if not is_audit_source(CODE / 'run_sampling_experiment.py'):
    sources = [p.parent for p in INPUTS.rglob('run_sampling_experiment.py') if is_audit_source(p)]
    if sources:
        shutil.copytree(sorted(sources)[0], CODE, dirs_exist_ok=True)
    else:
        matches = []
        archives = list(INPUTS.rglob('*.zip'))
        for archive in archives:
            if not zipfile.is_zipfile(archive): continue
            with zipfile.ZipFile(archive) as z:
                for member in z.namelist():
                    if member.endswith('/run_sampling_experiment.py') and marker.encode() in z.read(member):
                        matches.append((archive, member))
        if not matches:
            print('ZIP inputs found:', [str(p) for p in archives])
            raise FileNotFoundError('Updated audit code is not attached. Add BTP-code-sampling.zip, then rerun this cell.')
        archive, member = sorted(matches, key=lambda x: str(x[0]))[0]
        print('Using audit code:', archive)
        with tempfile.TemporaryDirectory(prefix='weak_audit_code_', dir='/kaggle/working') as tmp:
            staging = Path(tmp)
            with zipfile.ZipFile(archive) as z: safe_extract(z, staging)
            shutil.copytree((staging / member).parent, CODE, dirs_exist_ok=True)
assert is_audit_source(CODE / 'run_sampling_experiment.py'), 'Audit evaluator not found after extraction.'
import json, hashlib, math
import torch
expected = 'quarter_lr_to_epoch100_20260918_094320'
relative = expected + '/model/last.pt'
refs = list(INPUTS.rglob(relative))
if not refs:
    working = Path('/kaggle/working') / relative
    if working.is_file(): refs = [working]
if not refs:
    dest = Path('/kaggle/working/restored_quarter60')
    restored = dest / relative
    if restored.is_file(): refs = [restored]
    else:
        for archive in INPUTS.rglob('*.zip'):
            if not zipfile.is_zipfile(archive): continue
            with zipfile.ZipFile(archive) as z:
                if relative in z.namelist():
                    safe_extract(z, dest)
                    refs = [dest / relative]
                    break
assert len(refs) == 1, 'Attach quarter_lr_to_epoch100_20260918_094320.zip or its extracted folder.'
CHECKPOINT = refs[0]
for name in ['last.pt', 'best_iou.pth', 'best_psnr.pth']:
    assert CHECKPOINT.with_name(name).is_file(), f'Missing {name}'
REFERENCE = CHECKPOINT.parent.parent
assert (CODE / 'run_sampling_experiment.py').is_file()
DATA = Path('/kaggle/input/datasets/arnavnigamd/btp-data/public252')
assert torch.cuda.is_available(), 'Enable the GPU.'
print('Epoch100 reference:', REFERENCE)
subprocess.run([sys.executable,'verify_balanced_sampling.py'],cwd=CODE,check=True)


In [ ]:
from datetime import datetime
from IPython.display import display, FileLink
NAME = 'balanced_sampling_' + datetime.now().strftime('%Y%m%d_%H%M%S')
OUT = Path('/kaggle/working') / NAME
try:
    subprocess.run([sys.executable, '-u', 'run_sampling_experiment.py', '--root', str(DATA),
        '--reference', str(REFERENCE), '--out', str(OUT)], cwd=CODE, check=True)
finally:
    if OUT.exists():
        archive = shutil.make_archive(str(OUT), 'zip', root_dir=OUT.parent, base_dir=NAME)
        with zipfile.ZipFile(archive) as z:
            assert z.testzip() is None
            missing = [f'{NAME}/balanced50/model/{f}' for f in ['last.pt','best_iou.pth','best_psnr.pth']
                if f'{NAME}/balanced50/model/{f}' not in z.namelist()]
        print('Archive:', archive, 'Missing checkpoint files:', missing)
        import os
        os.chdir('/kaggle/working')
        display(FileLink(Path(archive).name))
        print('Download the complete ZIP before ending the session.')


In [ ]:
summary = json.loads((OUT / 'comparison_summary.json').read_text())
assert summary['arms']['balanced50']['final']['epoch']==110
for arm, result in summary['arms'].items():
    print(arm, json.dumps({k:v for k,v in result.items() if k != 'epochs'}, indent=2))
print('Historical control is reused from the completed loss experiment; only balanced50 was newly trained.')
